### 分析环境加载
- conda环境地址: /mnt/NexData_file/home/ztt/conda_env/spRNA

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '16'
import anndata
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import squidpy as sq
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from scipy.spatial import KDTree
from scipy.stats import mannwhitneyu, ttest_ind
import multiprocessing
from spatialdata_io import xenium
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context

import warnings
warnings.filterwarnings("ignore")

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
sc.settings.set_figure_params(dpi=100, dpi_save=200, figsize=(5, 5), facecolor='white')

### 输入数据、输出结果文件路径

注意：sample_info.txt文件要包含4列，分别是raw, fov, sample, group;
      如果一个fov中有多个样本，需要先圈选ROI,填入sample列，roi1.csv_name1这样，然后以,隔开。

| raw    	                                                        | fov                   | sample                               | group        |
|-------------------------------------------------------------------|-----------------------|--------------------------------------|--------------|
| output-XETG00305__0030260__lisiyuan__20250326__041009             | lisiyuan              | P9                                   | AGC          |
| output-XETG00305__0030260__liuweifei__20250326__041009            | liuweife              | P2                                   | EGC          |
| output-XETG00305__0030260__zhouchao_dingyongqing__20250326__041009| zhouchao_dingyongqing | zhouchao.csv_P10,dingyongqing.csv_P3 | AGC,EGC      |


In [ ]:
data_dir = '/data/xenium/FWSP20251139/'
sample_info = './sample_info.txt'
output_dir = './output'
organism = 'Human'

if not os.path.exists(output_dir):
    os.makedirs(os.path.join(output_dir, 'data'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'report'), exist_ok=True)

for i in ['01-Data_QC', 
          '02-Cell_filter', 
          '03-Dimension_reduction_cluster', 
          '04-Marker_gene_analysis', 
          '05-Cell_type_annotation', 
          '06-Computation_spatial_statistics',
          '07-Cell_Neighborhood_analysis',
          '08-Proximity_analysis',
          '09-Nearest_neighbor_analysis'
          ]:
    
    os.makedirs(os.path.join(output_dir, 'report', i), exist_ok=True)

### 多线程读数据

In [ ]:
# 多线程样本数据读入
def load_xenium_data(data_dir, sample_info, multi=False):
    def sd_read_xenium(raw_dir, fov_name, sample_name, sample_group, result_list):
        sdata = xenium(raw_dir, cells_as_circles=True, n_jobs=6)
        if hasattr(sdata, 'tables'):
            for name, table in sdata.tables.items():
                table.obs['fov'] = fov_name
                table.obs['sample'] = sample_name
                table.obs['group'] = sample_group
                result_list.append(table)                        

    with multiprocessing.Manager() as manager:
        result_list = manager.list()  # 使用Manager创建共享列表
        processes = []
        with open(sample_info, 'r') as f:
            line = f.readline()
            for line in f.readlines():
                line = line.strip().split('\t')
                raw_name = line[0]
                fov_name = line[1]
                sample_name = line[2]
                sample_group = line[3]
                p = multiprocessing.Process(
                        target=sd_read_xenium, 
                        args=(os.path.join(data_dir, raw_name), fov_name, sample_name, sample_group, result_list,)
                    )
                processes.append(p)
                p.start()

        for p in processes:
            p.join()

        adata = result_list[0].concatenate(result_list[1:], index_unique=None)
        adata.obs_names_make_unique()
        adata.var_names_make_unique()
        del adata.obs['batch']

        if multi:
            adata_single_sample = adata[~adata.obs['sample'].str.contains('.csv_')]
            adata_multi_sample = adata[adata.obs['sample'].str.contains('.csv_')]
            del adata_multi_sample.obs['sample']

            tmp = []
            sample2group = {}
            with open(sample_info, 'r') as f:
                line = f.readline()
                for line in f.readlines():
                    line = line.strip().split('\t')

                    if ',' in line[2] and ',' in line[3]:
                        sample2group.update(zip([i.split('.csv_')[1] for i in line[2].split(',')], line[3].split(',')))

                    if '.csv_' in line[2]:
                        roi = line[2].split(',')
                        for i in roi:
                            i = i.split('.csv_')
                            df = pd.read_csv(f'./ROI/{i[0]}.csv', skiprows=2)
                            df.rename(columns={'Cell ID': 'cell_id'}, inplace=True)
                            df['sample'] = i[1]
                            df['fov'] = line[1]
                            tmp.append(df[['cell_id', 'sample', 'fov']])

            all_roi = pd.concat(tmp, axis=0)

            merge_obs = pd.merge(adata_multi_sample.obs, all_roi, on=['cell_id', 'fov'], how='inner')

            ### 为了防止cell_id重复
            merge_obs['cell_id_fov'] = merge_obs['cell_id'].astype(str) + '-' + merge_obs['fov'].astype(str)
            adata_multi_sample.obs['cell_id_fov'] = adata_multi_sample.obs['cell_id'].astype(str) + '-' + adata_multi_sample.obs['fov'].astype(str)
            adata_multi_sample = adata_multi_sample[adata_multi_sample.obs['cell_id_fov'].isin(merge_obs['cell_id_fov'])]
            del adata_multi_sample.obs['cell_id_fov']
            ###

            adata_multi_sample.obs = merge_obs.copy()
            adata_all = adata_single_sample.concatenate(adata_multi_sample, index_unique=None)
            adata_all.obs['group'] = adata_all.obs.apply(lambda row: sample2group[row['sample']] if row['sample'] in sample2group else row['group'], axis=1)
            adata_all.obs['cell_id_sample'] = adata_all.obs['cell_id'].astype(str) + '-' + adata_all.obs['sample'].astype(str)
            del adata_all.obs['batch']
            return adata_all
        else:
            return adata

# 数据加载
adata_raw = load_xenium_data(data_dir=data_dir, sample_info='sample_info.txt', multi=True)
adata_raw.write(os.path.join(output_dir, 'data', 'adata_raw.h5ad'))

### 细胞质控、过滤

- data_qc: 下机数据质控，计算每个样本鉴定基因数、总counts数、线粒体基因占比、log10GenesPerUMI，并绘图；
- filter_by_quantile: 低质量细胞过滤；
- 根据样本自定义调整细胞鉴定基因数最小值和最大值、线粒体基因含量阈值、min_log10GenesPerUMI等参数

In [ ]:
def data_qc(adata, out_dir):
    
    def calculate_breaks(x):
        x = np.array(x)
        xx = np.round(x)   
        # Determine the steps based on the range
        if xx[1] < 100:
            scaled_times = np.ceil(100 / xx[1])
            steps = np.floor((scaled_times * xx[1] / 5) / 10) * 10 / scaled_times
        else:
            steps = np.floor(xx[1] / 5) / 10 * 10
            
        bk = np.arange(0, xx[1], steps)
        bk_minor = np.linspace(bk[0], bk[1], 5)
        bk = np.concatenate([bk_minor, bk[2:]])
        bk = bk[::-1]
        bk_minor = np.linspace(bk[0], bk[1], 5)
        bk = np.concatenate([bk_minor, bk[2:]])
        bk = bk[::-1]
        
        return [round(i,0) for i in bk]

    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 50, 100), inplace=True)
    adata.obs['log10GenesPerUMI'] = np.log10(adata.obs['n_genes_by_counts']) / np.log10(adata.obs['total_counts'])

    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'log10GenesPerUMI'], groupby='sample', jitter=False, rotation=90, multi_panel=True, show=False)
    axes = plt.gcf().get_axes()
    size = plt.gcf().get_size_inches()
    axes[0].grid(False)
    axes[1].grid(False)
    axes[2].grid(False)
    plt.savefig(os.path.join(out_dir, 'orig_nCount_nFeature_voilon.png'), bbox_inches = 'tight', pad_inches = 0.1)

    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))
    sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color="sample", title="", legend_loc='none', ax=ax1, show=False)
    sc.pl.scatter(adata[adata.obs['log10GenesPerUMI']>0.6], x='total_counts', y='log10GenesPerUMI', title="", color="sample", ax=ax2, show=False)
    ax1.grid(False)
    ax2.grid(False)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, 'orig_nCount_nFeature_scatter.png'), bbox_inches = 'tight')

    pd.DataFrame(adata.obs).to_csv(os.path.join(out_dir, 'all_cell_features_raw.xls'), sep='\t')

    for index, sample in enumerate(adata.obs['sample'].drop_duplicates()):
        out = os.path.join(out_dir, sample)
        os.makedirs(out, exist_ok=True)
        tmp = adata[adata.obs['sample']==sample]

        fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))
        sc.pl.scatter(tmp, x='total_counts', y='n_genes_by_counts', color="sample", title="", legend_loc='none', ax=ax1, show=False)
        sc.pl.scatter(tmp[tmp.obs['log10GenesPerUMI']>0.6], x='total_counts', y='log10GenesPerUMI', color="sample", title="", ax=ax2, show=False)
        ax1.grid(False)
        ax2.grid(False)
        plt.tight_layout()
        fig.savefig(os.path.join(out, 'orig_nCount_nFeature_scatter.png'), bbox_inches = 'tight')

        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(size[0], size[1]))
        breaks1 = calculate_breaks([tmp.obs['n_genes_by_counts'].min(), tmp.obs['n_genes_by_counts'].max()])
        breaks2 = calculate_breaks([tmp.obs['total_counts'].min(), tmp.obs['total_counts'].max()])  
        sc.pl.violin(tmp, 'n_genes_by_counts', xlabel="", ylabel="", jitter=0, groupby='sample', ax=axs[0], show=False)
        sc.pl.violin(tmp, 'total_counts', xlabel="", ylabel="", jitter=0, groupby='sample', ax=axs[1], show=False)
        sc.pl.violin(tmp[tmp.obs['log10GenesPerUMI']>0.6], 'log10GenesPerUMI', xlabel="", ylabel="", jitter=0, groupby='sample', ax=axs[2], show=False)
        axs[0].set_yticks(breaks1)
        axs[0].set_xlabel("")
        axs[0].set_title('nFeature_RNA')
        axs[1].set_yticks(breaks2)
        axs[1].set_xlabel("")
        axs[1].set_title('nCount_RNA')
        axs[2].set_xlabel("")
        axs[2].set_title('log10GenesPerUMI')
        axs[2].set_yticks([0.6, 0.8, 0.9, 0.95, 1])

        axs[0].grid(False)
        axs[1].grid(False)
        axs[2].grid(False)

        plt.tight_layout()
        plt.savefig(os.path.join(out, 'orig_nCount_nFeature_voilon'))

        pd.DataFrame(tmp.obs).to_csv(os.path.join(out, 'all_cell_features_raw.xls'), sep='\t')
        
    return adata
 
# QC
adata_raw = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_raw.h5ad'))
adata_raw.obs.reset_index(inplace=True)
adata_raw.obs.index = adata_raw.obs.index.astype(str)
adata_qc = data_qc(adata_raw, os.path.join(output_dir, 'report', '01-Data_QC'))
adata_qc.write(os.path.join(output_dir, 'data', 'adata_qc.h5ad'))

#### 低质量细胞过滤条件预设
- 生成各样本过滤参数用于后续低质量细胞过滤，可直接修改阈值

In [ ]:
def filter_parameter_record(adata, output_file):
    dic = {}
    for index, sample in enumerate(adata.obs['sample'].drop_duplicates()):
        tmp = adata[adata.obs['sample']==sample]
        min_genes = int(np.quantile(tmp.obs['n_genes_by_counts'], 0.05))
        max_genes = int(np.quantile(tmp.obs['n_genes_by_counts'], 0.995))
        dic.setdefault('sample', []).append(sample)
        dic.setdefault('min_genes', []).append(min_genes)
        dic.setdefault('max_genes', []).append(max_genes)
        dic.setdefault('min_GenesPerUMI', []).append(0.8)
        dic.setdefault('max_GenesPerUMI', []).append(0.99)

    pd.DataFrame(dic).to_csv(output_file, sep='\t', index=None)

adata_qc = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_qc.h5ad'))
filter_parameter_record(adata_qc, os.path.join(output_dir, 'filter_parameter.txt'))

In [ ]:
def cell_filter(adata_qc, para_file, out_dir):
    adata_list = []
    df_threshold = pd.read_csv(para_file, sep='\t')
    for index, row in df_threshold.iterrows():
        tmp = adata_qc[adata_qc.obs['sample']==row['sample']]
        tmp = tmp[
                    (tmp.obs['n_genes_by_counts'] > row['min_genes']) & \
                    (tmp.obs['n_genes_by_counts'] < row['max_genes']) & \
                    (tmp.obs['log10GenesPerUMI'] > row['min_GenesPerUMI']) & \
                    (tmp.obs['log10GenesPerUMI'] < row['max_GenesPerUMI'])
                 ]
        adata_list.append(tmp)
    adata = anndata.concat(adata_list, join='inner', index_unique=None)

    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'log10GenesPerUMI'], groupby='sample', jitter=False, rotation=90, multi_panel=True, show=False)
    axes = plt.gcf().get_axes()
    size = plt.gcf().get_size_inches()
    axes[0].grid(False)
    axes[1].grid(False)
    axes[2].grid(False)
    plt.savefig(os.path.join(out_dir, 'filtered_nCount_nFeature_voilon.png'), bbox_inches = 'tight', pad_inches = 0.1)

    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))
    sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color="sample", title="", legend_loc='none', ax=ax1, show=False)
    sc.pl.scatter(adata[adata.obs['log10GenesPerUMI']>0.6], x='total_counts', y='log10GenesPerUMI', title="", color="sample", ax=ax2, show=False)
    ax1.grid(False)
    ax2.grid(False)
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, 'filtered_nCount_nFeature_scatter.png'), bbox_inches = 'tight')

    pd.DataFrame(adata.obs).to_csv(os.path.join(out_dir, 'all_cell_features_filter.xls'), sep='\t')

    for index, sample in enumerate(adata.obs['sample'].drop_duplicates()):
        out = os.path.join(out_dir, sample)
        os.makedirs(out, exist_ok=True)
        tmp = adata[adata.obs['sample']==sample]

        sc.pl.violin(tmp, ['n_genes_by_counts', 'total_counts', 'log10GenesPerUMI'], groupby='sample', jitter=False, rotation=90, multi_panel=True, show=False)
        axes = plt.gcf().get_axes()
        axes[0].grid(False)
        axes[1].grid(False)
        axes[2].grid(False)
        plt.savefig(os.path.join(out, 'filtered_nCount_nFeature_voilon.png'), bbox_inches = 'tight', pad_inches = 0.1)

        fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))
        sc.pl.scatter(tmp, x='total_counts', y='n_genes_by_counts', color="sample", title="", legend_loc='none', ax=ax1, show=False)
        sc.pl.scatter(tmp, x='total_counts', y='log10GenesPerUMI', color="sample", title="", ax=ax2, show=False)
        ax1.grid(False)
        ax2.grid(False)
        plt.tight_layout()
        fig.savefig(os.path.join(out, 'filtered_nCount_nFeature_scatter.png'), bbox_inches = 'tight')

    return adata

adata_filter = cell_filter(adata_qc, para_file=os.path.join(output_dir, 'filter_parameter.txt'), out_dir=os.path.join(output_dir, 'report', '02-Cell_filter'))
adata_filter.write(os.path.join(output_dir, 'data', 'adata_filtered.h5ad'))

降维聚类

- 这里由于数据量大，降维聚类使用GPU进行计算，需要加载scanpy-gpu环境

- 打开cluster_gpu.ipynb先降维聚类

特征基因筛选、细胞类型定义

In [ ]:
def cluster_plot(adata, res, out_dir):
    with rc_context({'figure.figsize': (6, 6)}):
        sc.pl.umap(
            adata, 
            color=res,  
            title='Cell Cluster', 
            legend_loc='on data', 
            size=8,
            legend_fontsize=9, 
            legend_fontweight='normal', 
            legend_fontoutline=2,
            show=False
        )
        plt.savefig(os.path.join(out_dir, 'umap_dimplot_cluster.png'), facecolor='white', bbox_inches="tight", dpi=300)
        plt.savefig(os.path.join(out_dir, 'umap_dimplot_cluster.pdf'), facecolor='white', bbox_inches="tight")

    library_names = adata.obs['sample'].drop_duplicates().to_list()
    for library in library_names:
        with rc_context({'figure.figsize': (8, 8)}):
            sq.pl.spatial_scatter(
                adata[adata.obs['sample'] == library,:],
                library_id="spatial",
                color=[res],
                title=f'{library}',
                shape=None,
                size=1.5,
                img=False,
                use_raw=True
            )
            plt.savefig(os.path.join(out_dir, f'spatial_{library}_cluster.png'), facecolor='white', bbox_inches="tight", dpi=300)
            plt.savefig(os.path.join(out_dir, f'spatial_{library}_cluster.pdf'), facecolor='white', bbox_inches="tight")

    groups = adata.obs['group'].drop_duplicates().tolist()
    if len(groups) > 1:
        axs = tuple(map(lambda x: f'ax{x+1}', range(len(groups))))
        fig, axs = plt.subplots(nrows=1, ncols=len(groups), figsize=(len(groups)*5, 5))
        for i in range(len(groups)):
            if i == len(groups) - 1:
                legend = 'right margin'
            else:
                legend = None
            sc.pl.umap(
                adata[adata.obs["group"]==groups[i]], 
                color=res, 
                title=groups[i], 
                ax=axs[i], 
                wspace=0.1, 
                size=8,
                legend_loc='on data', 
                legend_fontsize=12, 
                legend_fontweight='normal', 
                legend_fontoutline=2, 
                show=False
            )
        plt.savefig(os.path.join(out_dir, 'umap_split_group_cluster.png'), facecolor='white', bbox_inches="tight", dpi=300)
        plt.savefig(os.path.join(out_dir, 'umap_split_group_cluster.pdf'), facecolor='white', bbox_inches="tight")

    result = adata.obs.groupby([res, 'sample']).size().reset_index(name='number').pivot(index=res, columns='sample', values='number').reset_index()
    result.columns.name = None
    # result['Total'] = result.sum(axis=1)
    # result['Percent'] = round((result['Total'] / sum(result['Total'])) * 100, 1)
    result.to_csv(os.path.join(out_dir, 'cluster_statistics.xls'), sep='\t', index=None)
    adata.obs.to_csv(os.path.join(out_dir, 'all_cluster_cells_features.xls'), sep='\t')
    pd.DataFrame(adata.obsm['X_umap'], index=adata.obs_names, columns=['UMAP_1', 'UMAP_2']).to_csv(os.path.join(out_dir, 'data_umap.xls'), sep='\t')

    ct_by_sample=pd.crosstab(adata.obs['sample'], adata.obs[res])
    cts=ct_by_sample.div(ct_by_sample.sum(axis=1),axis=0)
    cls_dic=dict(zip(np.unique(adata.obs[res]), adata.uns[f'{res}_colors']))

    cts.plot(
        kind='bar',
        stacked=True,
        color=cts.columns.map(cls_dic),
        legend=False,
        figsize=(len(cts.index)*0.75, 4.5),
        width=0.98
    )
    plt.grid(False)
    plt.title('Cluster cell proportions')
    plt.legend(bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0, ncol=2)
    plt.savefig(os.path.join(out_dir, 'cluster_cell_proportion.png'), facecolor='white', bbox_inches="tight")
    plt.savefig(os.path.join(out_dir, 'cluster_cell_proportion.pdf'), facecolor='white', bbox_inches="tight")

    return adata


def marker_gene(adata, res, organism, out_dir):

    def go_enrichment(gene_list, organism, out_dir):
        if organism == 'Human':
            background = 'hsapiens_gene_ensembl'
        else:
            background = 'mmusculus_gene_ensembl'
            
        enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets=['GO_Biological_Process_2023', 'GO_Cellular_Component_2023', 'GO_Molecular_Function_2023'],
            background=background,
            organism=organism, 
            cutoff=0.05,
            no_plot=True,
            outdir=out_dir,
            verbose=False
        )

        enr.results['Term'] = enr.results['Term'].str.replace(r' \(.*?\)', '', regex=True)
        enr.results['Gene_set'] = enr.results['Gene_set'].replace('GO_Biological_Process_2023', "BP").replace("GO_Cellular_Component_2023","CC").replace("GO_Molecular_Function_2023", "MF")
        enr.results.rename(columns={"Gene_set": "GO"}, inplace=True)
        
        gp.barplot(
            enr.results,
            column="Adjusted P-value",
            group='GO', # set group, so you could do a multi-sample/library comparsion
            size=10,
            top_term=5,
            figsize=(3,5),
            title="GO Enrichment",
            color=['darkgreen', 'darkblue', 'darkred'],
            ofname=os.path.join(out_dir, 'GO_Enrichment.png')
        )

    sc.tl.rank_genes_groups(
        adata, 
        groupby=res, 
        use_raw=True, 
        method="t-test", # method='t-test','wilcoxon',key_added='rank_genes_groups' 
        pts=True
    )
    
    sc.pl.rank_genes_groups_dotplot(
        adata, 
        groupby=res, 
        standard_scale="var", 
        n_genes=5,
        cmap='bwr',
        show=False,
    )
    plt.savefig(os.path.join(out_dir, 'dotplot_top5_marker_gene.png'), facecolor='white', bbox_inches="tight", dpi=200)

    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    all_diff = []
    for c in groups:
        out = os.path.join(out_dir, f'cluster{c}')
        os.makedirs(out, exist_ok=True)
        tmp = sc.get.rank_genes_groups_df(adata, group=c)
        tmp = tmp[(tmp['logfoldchanges']>0) & (tmp['pvals_adj']<0.05)]
        tmp.to_csv(os.path.join(out, 'differentially_expressed_genes.xls'), sep='\t', index=None)
        
        try:
            go_enrichment(tmp['names'], organism=organism, out_dir=out)
        except:
            print(tmp)

        tmp['cluster'] = c
        all_diff.append(tmp)

    all_diff = pd.concat(all_diff, axis=0)
    all_diff.rename(columns={'names': 'Symbol'}, inplace=True)
    all_diff.to_csv(os.path.join(out_dir, 'all_cluster_marker.xls'), sep='\t', index=None)
    all_diff['cluster'].value_counts().sort_index().to_csv(os.path.join(out_dir, 'cluster_diff_statistics.xls'), sep='\t')
    
    return adata


In [ ]:
res = 'res_0.3'
adata = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_cluster.h5ad'))
adata = cluster_plot(adata, res, out_dir=os.path.join(output_dir, 'report', '03-Dimension_reduction_cluster'))
adata = marker_gene(adata, res, organism=organism, out_dir=os.path.join(output_dir, 'report', '04-Marker_gene_analysis'))

细胞类型定义

In [ ]:
def celltype_anno(adata, res, anno_dict, out_dir):
    adata.obs['celltype'] = [anno_dict[clust] for clust in adata.obs[res]]    
    with rc_context({'figure.figsize': (6, 6)}):
        sc.pl.umap(
            adata, 
            color='celltype', 
            show=False, 
            title='Cell Types', 
            legend_loc='right margin', 
            legend_fontsize=9, 
            size=8, 
            legend_fontweight='normal', 
            legend_fontoutline=2
        )
        plt.savefig(os.path.join(out_dir, 'umap_dimplot_celltype.png'), facecolor='white', bbox_inches="tight", dpi=300)

    groups = adata.obs['group'].drop_duplicates().tolist()
    if len(groups) > 1:
        axs = tuple(map(lambda x: f'ax{x+1}', range(len(groups))))
        fig, axs = plt.subplots(ncols=len(groups), figsize=(len(groups)*6, 6))
        for i in range(len(groups)):
            sc.pl.umap(
                adata[adata.obs["group"]==groups[i]],
                color="celltype", 
                title=groups[i], 
                ax=axs[i], 
                wspace=0.1, 
                size=10,
                legend_loc='on data', 
                legend_fontsize=12, 
                legend_fontweight='normal', 
                legend_fontoutline=2, 
                show=False
            )
        plt.savefig(os.path.join(out_dir, 'umap_dimplot_celltype_groups.png'), facecolor='white', bbox_inches="tight", dpi=300)

    pd.DataFrame(adata.obs).to_csv(os.path.join(out_dir, 'all_cell_features_celltype.xls'), sep='\t')

    library_names = adata.obs['sample'].drop_duplicates().to_list()
    for library in library_names:
        with rc_context({'figure.figsize': (8, 8)}):
            sq.pl.spatial_scatter(
                adata[adata.obs['sample'] == library,:],
                library_id="spatial",
                color=["celltype"],
                title=f'{library}',
                shape=None,
                size=1.5,
                img=False,
                use_raw=True
            )
            plt.savefig(os.path.join(out_dir, f'spatial_{library}_celltype.png'), facecolor='white', bbox_inches="tight", dpi=300)

    ct_by_sample=pd.crosstab(adata.obs['sample'], adata.obs['celltype'])
    cts=ct_by_sample.div(ct_by_sample.sum(axis=1),axis=0)
    cls_dic=dict(zip(np.unique(adata.obs['celltype']), adata.uns['celltype_colors']))

    cts.plot(
        kind='bar',
        stacked=True,
        color=cts.columns.map(cls_dic),
        legend=False,
        figsize=(len(cts.index)*1.2, 4),
        width=0.98
    )
    plt.grid(False)
    plt.title('Cell type proportions')
    plt.legend(bbox_to_anchor=(1.02, 1), loc=2, borderaxespad=0)
    plt.savefig(os.path.join(out_dir, 'celltype_proportion.png'), facecolor='white', bbox_inches="tight", dpi=300)
    
    return adata

#############

major_annot = {
    "0": "Plasma",
    "1": "Macrophage",
    "2": "Basal cells",
    "3": "Neutrophi",
    "4": "Basal cells",
    "5": "T cells",
    "6": "Fibroblasts",
    "7": "Fibroblasts",
    "8": "Neutrophi"
}

adata = celltype_anno(
    adata,
    res=res, 
    anno_dict=major_annot, 
    out_dir=os.path.join(output_dir, 'report', '05-Cell_type_annotation')
)

adata.write(os.path.join(output_dir, 'data', 'adata_celltype.h5ad'))

中心性得分

In [ ]:
def spatial_statistics(adata, out_dir):
    for group in adata.obs['group'].drop_duplicates():
        tmp = adata[adata.obs['group'] == group]
        sq.gr.spatial_neighbors(tmp, coord_type="generic", n_neighs=10, delaunay=True)
        sq.gr.centrality_scores(tmp, cluster_key="celltype")
        sq.pl.centrality_scores(tmp, cluster_key="celltype", figsize=(16, 5))
        plt.savefig(os.path.join(out_dir, f'{group}_centrality_scores_plot.png'), facecolor='white', bbox_inches="tight", dpi=300)
        plt.savefig(os.path.join(out_dir, f'{group}_centrality_scores_plot.pdf'), facecolor='white', bbox_inches="tight")

adata = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_celltype.h5ad'))
spatial_statistics(adata, out_dir=os.path.join(output_dir, 'report', '06-Computation_spatial_statistics'))

CN分析

In [ ]:
def cell_neighbors(adata, column, n_neighbors=10, k=10, n_clusters=10, out_dir=None):
    spatial_coords = adata.obsm['spatial']
    onehot_encoding = pd.get_dummies(adata.obs[column])
    
    # 计算最近邻居
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(spatial_coords)
    distances, indices = nbrs.kneighbors(spatial_coords)

    # 对每个cell，计算其邻居的类型分布
    cluster_cols = adata.obs[column].unique()
    values = onehot_encoding[cluster_cols].values

    def compute_window_sums(indices, values, k):
        windows = []
        for idx in range(indices.shape[0]):
            neighbors = indices[idx, :k]
            window_sum = values[neighbors].sum(axis=0)
            windows.append(window_sum)
        return np.array(windows)
    
    windows = compute_window_sums(indices, values, k)
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=0)
    labels = km.fit_predict(windows)
    adata.obs[f'CNs_{n_clusters}'] = [f'CN{i}' for i in labels]

    adata.obs.to_csv(os.path.join(out_dir, f'CNs_{n_clusters}_all_cell_features.xls'), sep='\t', index=None)

    # 保存聚类中心
    k_centroids = km.cluster_centers_
    
    # 计算Fold Change (fc)
    tissue_avgs = values.mean(axis=0)
    fc = np.log2(((k_centroids + tissue_avgs) / (k_centroids + tissue_avgs).sum(axis=1, keepdims=True)) / tissue_avgs)  
    fc = pd.DataFrame(fc, columns=adata.obs[column].unique())
    fc.index = 'CN' + fc.index.astype(str)
    
    plt.figure(figsize=(6, 4))
    sns.set_style("white")
    sns.clustermap(fc, vmin=-2, vmax=2, cmap="coolwarm", row_cluster=False, linewidths=0)
    plt.savefig(os.path.join(out_dir, f"CNs_{n_clusters}.png"), dpi=300, bbox_inches='tight')
    
    # 统计柱状图
    # 计算每个样本中每个聚类的细胞数量
    cluster_counts = adata.obs.groupby(['sample', f'CNs_{n_clusters}']).size().unstack(fill_value=0)

    # 计算每个样本中每个聚类的细胞占比
    cluster_proportions = cluster_counts.div(cluster_counts.sum(axis=1), axis=0)

    # 转换为长格式以便绘图
    cluster_proportions_long = cluster_proportions.reset_index().melt(id_vars='sample', var_name='Cluster', value_name='Proportion')

    # 添加组信息
    cluster_proportions_long = cluster_proportions_long.merge(adata.obs[['sample', 'group']].drop_duplicates(), on='sample')

    # 绘制箱线图
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='Cluster', y='Proportion', hue='group', data=cluster_proportions_long)
    sns.stripplot(x='Cluster', y='Proportion', hue='group', data=cluster_proportions_long, dodge=True, jitter=False, color='gray', alpha=0.3, legend=False)
    plt.savefig(os.path.join(out_dir, f"CNs_{n_clusters}_cell_proportion.png"), dpi=300, bbox_inches='tight')

    library_names = adata.obs['sample'].drop_duplicates()
    for library in library_names:
        with rc_context({'figure.figsize': (8, 8)}):
            sq.pl.spatial_scatter(
                adata[adata.obs['sample'] == library,:],
                library_id="spatial",
                color=[f'CNs_{n_clusters}'],
                title=f'{library} CNs',
                shape=None,
                # size=1,
                img=False,
                use_raw=True
            )
            plt.savefig(os.path.join(out_dir, f'{library}_spatial_CNs_{n_clusters}.png'), facecolor='white', bbox_inches="tight", dpi=300)

    return adata


adata = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_celltype.h5ad'))

adata = cell_neighbors(
    adata,
    'celltype', 
    n_neighbors=10, 
    k=10, 
    n_clusters=10, 
    out_dir=os.path.join(output_dir, 'report', '07-Cell_Neighborhood_analysis')
)

adata = cell_neighbors(
    adata,
    'celltype', 
    n_neighbors=10, 
    k=10, 
    n_clusters=15, 
    out_dir=os.path.join(output_dir, 'report', '07-Cell_Neighborhood_analysis')
)

邻近度分析
- 各细胞类型周围50um细胞组成
- 各细胞类型周围(0, 50, 100, 150, 200um)细胞组成

这里可以继续优化，改成多任务并行提升速度

In [ ]:
def proximity_analysis(adata, out_dir):   
    ## 计算50um邻近度
    distance_threshold = 50
    sq.gr.spatial_neighbors(adata, coord_type='generic', radius=50)
    sq.gr.interaction_matrix(adata, cluster_key='celltype')
    group_list = adata.obs['group'].unique()
    palette = dict(zip(group_list, sns.color_palette("Set2", n_colors=len(group_list))))

    ## 无统计检验
    for ct in adata.obs['celltype'].unique():
        out = os.path.join(out_dir, ct)
        os.makedirs(out, exist_ok=True)
        proximity_data = []
        cells_of_type = adata.obs_names[adata.obs['celltype'] == ct]
        for cell_idx in adata.obs_names.get_indexer(cells_of_type):
            neighbors = adata.obsp['spatial_connectivities'][cell_idx].nonzero()[1]
            neighbor_types = adata.obs.loc[adata.obs.index[neighbors], 'celltype']
            counts = neighbor_types.value_counts(normalize=True)
            for neighbor_type, proportion in counts.items():
                proximity_data.append({
                    'celltype': neighbor_type,
                    'proportion': proportion,
                    'sample': adata.obs.loc[adata.obs.index[cell_idx], 'sample'],
                    'group': adata.obs.loc[adata.obs.index[cell_idx], 'group']
                })

        results_df = pd.DataFrame(proximity_data)
        results_df.to_csv(os.path.join(out, f'nearly_{distance_threshold}um_distance_celltype_proportion.xls'), sep='\t', index=None)

        plt.figure(figsize=(10, 6))
        ax = sns.boxplot(
            x='celltype', 
            y='proportion', 
            hue='group', 
            palette=palette,
            data=results_df, 
            showfliers=False
        )
        ax.grid(visible=False)

        max_proportion = results_df['proportion'].max()
        # max_proportion = 1
        # # y_offset = max_proportion * 0.1

        # 进行统计检验并添加 p 值
        celltypes = results_df['celltype'].unique()
        for celltype in celltypes:
            data_subset = results_df[results_df['celltype'] == celltype]
            groups = data_subset['group'].unique()       
            if len(groups) == 2:  # 确保只有两个组
                group1 = data_subset[data_subset['group'] == groups[0]]['proportion']
                group2 = data_subset[data_subset['group'] == groups[1]]['proportion']

                # 使用 Mann-Whitney U 检验计算 p 值
                if len(group1) > 0 and len(group2) > 0:
                    stat, p_value = mannwhitneyu(group1, group2)

                    # 判断显著性并标注
                    if p_value < 0.01:
                        annotation = '**'
                    elif p_value < 0.05:
                        annotation = '*'
                    else:
                        annotation = 'ns'

                    # 添加注释，标注显著性水平，仅显示符号，放在图形顶部
                    x_position = list(celltypes).index(celltype)
                    ax.annotate(f'{annotation}', 
                                xy=(x_position, max_proportion-max_proportion * 0.2),
                                ha='center', color='black', fontsize=12)

            # ax.set_ylim(0, max_proportion * 1.2)
        
        ax.legend(title="", bbox_to_anchor=(1.02, 0.5), loc='upper left', borderaxespad=0.)
        
        plt.xlabel("")
        plt.title(f'Celltype proportion of {ct} nearly {distance_threshold}um distance')
        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(os.path.join(out, 'proximity_50um.png'))
        plt.savefig(os.path.join(out, 'proximity_50um.pdf'))
        plt.clf()

############################################################################################################

    # 获取细胞坐标和细胞类型
    coords = adata.obsm['spatial']
    cell_types = adata.obs['celltype']
    groups = adata.obs['group']

    # 创建 KDTree
    tree = KDTree(coords)

    # 定义距离范围
    distance_ranges = [0, 50, 100, 150, 200]

    for ct in np.unique(cell_types):
        out = os.path.join(out_dir, ct)
        os.makedirs(out, exist_ok=True)

        proximity_data = []
        b_cells_indices = np.where(cell_types == ct)[0]
        for b_idx in b_cells_indices:
            b_coord = coords[b_idx]
            
            # 查询邻居在各个距离范围内的细胞
            previous_indices = set()
            for i in range(len(distance_ranges) - 1):
                # 获取在当前最大距离范围内的邻居
                indices_within_max_range = set(tree.query_ball_point(b_coord, distance_ranges[i + 1]))
                
                # 去掉自己
                indices_within_max_range.discard(b_idx)
                
                # 当前范围内的邻居是最大范围内的邻居减去之前的邻居
                current_indices = indices_within_max_range - previous_indices
                
                # 更新之前的邻居集合
                previous_indices.update(current_indices)
                
                # 统计当前范围内的邻居类型
                neighbor_types = cell_types[list(current_indices)]
                counts = pd.Series(neighbor_types).value_counts()
                
                # 将计数结果存储到 proximity_data 中，并计算均值
                for neighbor_type in counts.index:
                    proximity_data.append({
                        'distance_range': f"{distance_ranges[i]}-{distance_ranges[i + 1]}",
                        'celltype': neighbor_type,
                        'count': counts[neighbor_type],
                        'group': groups[b_idx]
                    })

        # 转换为 DataFrame
        proximity_df = pd.DataFrame(proximity_data)

        # 按照 distance_range 和 celltype 分组，并计算均值
        mean_proximity_df = proximity_df.groupby(['distance_range', 'celltype', 'group'], as_index=False).median()

        # Ensure correct order of distance ranges
        mean_proximity_df['distance_range'] = pd.Categorical(
            mean_proximity_df['distance_range'], 
            categories=["0-50", "50-100", "100-150", "150-200"], 
            ordered=True
        )
        mean_proximity_df.sort_values('distance_range', inplace=True)

        # 绘制折线图，按细胞类型分面展示，分成4列
        celltypes = sorted(mean_proximity_df['celltype'].unique().tolist())
        n_cols = 4  # 每行显示4个子图
        n_rows = (len(celltypes) + n_cols - 1) // n_cols  # 计算行数

        fig, axs = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4), sharex=True)

        for ax, celltype in zip(axs.flatten(), celltypes):
            subset = mean_proximity_df[mean_proximity_df['celltype'] == celltype]
            for group in sorted(subset['group'].unique().tolist()):
                group_data = subset[subset['group'] == group]
                ax.plot(group_data['distance_range'], group_data['count'], marker='o', label=group)
            
            ax.set_title(f'{celltype}')
            ax.set_ylabel('Median cell count')
            # ax.set_xlabel('Distance (um)')
            # ax.grid()
            ax.legend()

        # 隐藏多余的子图（如果有）
        for i in range(len(celltypes), n_rows * n_cols):
            fig.delaxes(axs.flatten()[i])

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(os.path.join(out, 'proximity_distance_range.png'))
        plt.savefig(os.path.join(out, 'proximity_distance_range.pdf'))
        plt.clf()

adata_celltype = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_celltype.h5ad'))
proximity_analysis(adata=adata_celltype, out_dir=os.path.join(output_dir, 'report', '08-Proximity_analysis')) 

### 最近邻分析

In [ ]:
def nearest_neighbor(adata, out_dir):
    coords = adata.obsm['spatial']
    tree = KDTree(coords)
    distances, indices = tree.query(coords, k=2)

    group_list = adata.obs['group'].unique()
    palette = dict(zip(group_list, sns.color_palette("Set2", n_colors=len(group_list))))

    groups = adata.obs['group'].values
    samples = adata.obs['sample'].values
    cell_types = adata.obs['celltype'].values
    for cell_type in np.unique(cell_types):
        out = os.path.join(out_dir, cell_type)
        os.makedirs(out, exist_ok=True)
        proportions = []
        for i, current_cell_type in enumerate(cell_types):
            if current_cell_type == cell_type:
                neighbor_types = cell_types[indices[i, 1:]]  # 排除自己
                neighbor_groups = groups[indices[i, 1:]]  # 获取邻居的分组信息
                neighbor_samples = samples[indices[i, 1:]]
                neighbor_df = pd.DataFrame({'source_type': cell_type, 'neighbor_type': neighbor_types, 'distance': distances[i, 1:], 'sample': neighbor_samples, 'group': neighbor_groups})   
                proportions.append(neighbor_df)

        df = pd.concat(proportions)
        df.to_csv(os.path.join(out, 'nearest_neighbor_proportion.xls'), sep='\t', index=None)


        # 使用 FacetGrid 进行分面绘图
        g = sns.FacetGrid(df, col="neighbor_type", hue="group", col_wrap=4, palette=palette)
        g.map(sns.kdeplot, "distance", multiple="stack", alpha=0.5, linewidth=0)

        # 设置子图的标题
        g.set_titles("{col_name}", size=14)

        # 隐藏x轴title
        # g.set_axis_labels("", "")

        # 移动图例到右上角并调整格式
        g.add_legend(title="Group", bbox_to_anchor=(1, 1), loc='upper left')

        # 强制显示所有子图的 Y 轴刻度和标签
        for ax in g.axes.flatten():
            ax.yaxis.set_tick_params(labelleft=True)  # 强制显示 y 轴刻度标签
            ax.tick_params(axis='y', which='both', left=True)  # 显示 y 轴刻度
            ax.grid(False)
            ax.set_xlim(0, 30)
            # ax.set_xticks([])

        # 调整子图之间的间距
        g.fig.subplots_adjust(wspace=0.3)

        g.savefig(os.path.join(out, f"nearest_neighbor_proportion.png"), dpi=300, bbox_inches='tight')


def nearest_neighbor_boxplot(adata, out_dir):
    sns.set_theme(style="white")
    cell_types = adata.obs['celltype'].values

    group_list = adata.obs['group'].unique()
    palette = dict(zip(group_list, sns.color_palette("Set2", n_colors=len(group_list))))
    
    for cell_type in np.unique(cell_types):
        out = os.path.join(out_dir, cell_type)
        os.makedirs(out, exist_ok=True)
    
        # 读取数据
        df = pd.read_csv(os.path.join(out, 'nearest_neighbor_proportion.xls'), sep='\t')
        
        # 创建按 neighbor_type 分面的 FacetGrid，并设置每行 4 列
        g = sns.FacetGrid(df, col="neighbor_type", col_wrap=4, aspect=1)

        # 在每个分面上绘制嵌套的箱线图，展示 group 与 distance 的关系
        g.map_dataframe(sns.boxplot, x="group", y="distance", hue="group", palette=palette, width=0.6)

        # 手动计算 p 值并添加到每个子图上
        groups = df['group'].unique()

        overall_max_distance = int(max(df['distance']))-1
        
        overall_max_distance = 14

        # 遍历每个分面
        for ax, (neighbor_type, data_subset) in zip(g.axes.flat, df.groupby("neighbor_type")):
            if len(groups) == 2:  # 假设 group 列只有两个组
                group1 = data_subset[data_subset['group'] == groups[0]]['distance']
                group2 = data_subset[data_subset['group'] == groups[1]]['distance']

                # 使用 Mann-Whitney U 检验计算 p 值
                if len(group1) > 0 and len(group2) > 0:  # 确保每组有数据
                    stat, p_value = mannwhitneyu(group1, group2)
                    
                    # 判断显著性并标注
                    if p_value < 0.01:
                        annotation = '**'
                    elif p_value < 0.05:
                        annotation = '*'
                    else:
                        annotation = 'ns'

                    # 添加注释，标注显著性水平，仅显示符号，放在图的最上边
                    ax.annotate(f'{annotation}', 
                                xy=(0.5, overall_max_distance),  # 位置设置在图形顶部
                                ha='center', color='black', fontsize=12)

            ax.set_ylim(0, 15)

        g.set_axis_labels("", "median cell counts")
        g.set_titles("{col_name}")

        g.savefig(os.path.join(out, f"nearest_neighbor_proportion_boxplot.png"), dpi=300, bbox_inches='tight')


adata = sc.read_h5ad(os.path.join(output_dir, 'data', 'adata_celltype.h5ad'))
nearest_neighbor(adata, os.path.join(output_dir, 'report', '09-Nearest_neighbor_analysis'))        
nearest_neighbor_boxplot(adata, os.path.join(output_dir, 'report', '09-Nearest_neighbor_analysis'))        

生成标准分析报告
- 注意修改项目编号、客户信息等

In [ ]:
def make_report(pinfo):
    report_dir = os.path.join(output_dir, 'report')
    os.system(f'cp -r /mnt/NexData_file/home/ztt/analysis_pipeline/report_template/xenium_template/* {report_dir}')
    rmd_template = os.path.join(report_dir, 'report.rmd')

    with open(rmd_template, 'r', encoding='utf-8') as f:
        line = f.read()
        rmd = line.replace('%pid%', pinfo['pid']).replace('%pname%', pinfo['pname']).replace('%paddress%', pinfo['paddress']).replace('%pcustmor%', pinfo['pcustmor']).replace('%pnum%', pinfo['pnum'])

    with open(rmd_template, 'w', encoding='utf-8') as w:
        w.write(rmd)

    os.system("R -e \"rmarkdown::render('{0}/report.rmd',output_file='{0}/report.html')\"".format(os.path.abspath(report_dir)))

pinfo = {
    'pid': '项目编号',
    'pname': '项目名称',
    'paddress': '客户单位',
    'pcustmor': '客户姓名',
    'pnum': '样本数',
}

make_report(pinfo)